# Notebook 2: Baseline Models
**MarathiMWP Thesis — Chapter 4 (Methodology) + Chapter 5 (Results)**

Implements three baseline approaches:
1. **Rule-Based**: Marathi keyword → operation mapping
2. **Template-Based**: Pattern matching to equation templates
3. **Seq2Seq LSTM**: Character-level encoder-decoder

**Prerequisites**: Run Notebook 01 first to generate splits.

In [2]:
# !pip install torch sacrebleu tqdm

import sys, json, re, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import Counter

sys.path.insert(0, str(Path('.').resolve()))
from utils.data_utils import load_splits, extract_numbers, classify_operation
from utils.evaluation import compute_metrics, print_metrics, evaluate_equation

random.seed(42)
np.random.seed(42)

Path('figures').mkdir(exist_ok=True)
print('Setup complete')

Setup complete


In [3]:
train, val, test = load_splits('marathi')
print(f'Train: {len(train)}  Val: {len(val)}  Test: {len(test)}')

gold_test_eqs = [x['Equation'] for x in test]

Train: 1634  Val: 349  Test: 353


## 1. Evaluation Metrics Setup

In [4]:
# Quick sanity check on evaluation
from utils.evaluation import answers_match, equations_exact_match, sentence_bleu

g = 'X = ( 7.423 + 6.129 )'
p = 'X = ( 6.129 + 7.423 )'   # commutative
print(f'Exact match (commutative): {equations_exact_match(g, p)}')  # False — expected
print(f'Answer match (commutative): {answers_match(g, p)}')          # True — expected

g2 = 'X = ( 47 - 25 )'
p2 = 'X = ( 47 - 25 )'
print(f'Exact match (identical): {equations_exact_match(g2, p2)}')    # True

Exact match (commutative): False
Answer match (commutative): True
Exact match (identical): True


## 2. Rule-Based Baseline

Strategy:
- Extract numbers from the Marathi problem text
- Use keyword lookup to identify the mathematical operation
- Apply heuristic ordering rules (e.g., for subtraction: larger - smaller)

Expected accuracy: ~15-25% (simple heuristics, no learning)

In [5]:
# Marathi operation keywords
ADDITION_KW    = ['बेरीज', 'जोड', 'एकत्र', 'मिळून', 'एकूण', 'आणखी', 'मिळाले', 'जास्त', 'वाढ']
SUBTRACTION_KW = ['वजा', 'कमी', 'शिल्लक', 'उरले', 'काढले', 'फरक', 'गेले', 'खर्च', 'दिले']
MULTIPLY_KW    = ['गुणा', 'पट', 'प्रत्येक', 'गुणिले']
DIVISION_KW    = ['भाग', 'भागा', 'वाटा', 'समान', 'प्रत्येकाला']


def detect_operation(text: str) -> str:
    """Return guessed operation from Marathi keyword presence."""
    scores = {'addition': 0, 'subtraction': 0, 'multiplication': 0, 'division': 0}
    for kw in ADDITION_KW:
        if kw in text:
            scores['addition'] += 1
    for kw in SUBTRACTION_KW:
        if kw in text:
            scores['subtraction'] += 1
    for kw in MULTIPLY_KW:
        if kw in text:
            scores['multiplication'] += 1
    for kw in DIVISION_KW:
        if kw in text:
            scores['division'] += 1

    best = max(scores, key=scores.get)
    return best if scores[best] > 0 else 'addition'  # default fallback


def rule_based_solve(problem: dict) -> str:
    text = problem['Problem']
    numbers = extract_numbers(text)

    # Use relevant indices to pick numbers when available
    rel_idx = problem.get('Relevant Indices', [])
    relevant_nums = []
    for idx in rel_idx:
        if idx != 'implicit' and isinstance(idx, int) and idx < len(numbers):
            relevant_nums.append(numbers[idx])
    if not relevant_nums:
        relevant_nums = numbers[:2] if len(numbers) >= 2 else numbers

    if len(relevant_nums) == 0:
        return 'X = 0'

    if len(relevant_nums) == 1:
        return f'X = {relevant_nums[0]}'

    a, b = relevant_nums[0], relevant_nums[1]
    op = detect_operation(text)

    if op == 'addition':
        return f'X = ( {a} + {b} )'
    elif op == 'subtraction':
        # Heuristic: larger - smaller
        big, small = (a, b) if a >= b else (b, a)
        return f'X = ( {big} - {small} )'
    elif op == 'multiplication':
        return f'X = ( {a} * {b} )'
    elif op == 'division':
        if b != 0:
            return f'X = ( {a} / {b} )'
        return f'X = ( {a} / {b} )'

    return f'X = ( {a} + {b} )'


# Test on one example
ex = test[0]
print('Problem:', ex['Problem'])
print('Gold    :', ex['Equation'])
print('Pred    :', rule_based_solve(ex))

Problem: एक शर्ट शिवण्यासाठी 4 मीटर कापड आवश्यक आहे. 45 मीटर कापडापासून शर्ट शिवल्यानंतर किती कापड शिल्लक राहील?
Gold    : X = ( 45 % 4 )
Pred    : X = 45.0


In [7]:
# Evaluate rule-based on test set
rule_preds = [rule_based_solve(x) for x in test]
rule_metrics = compute_metrics(gold_test_eqs, rule_preds)
print_metrics('Rule-Based Baseline', rule_metrics)


  Rule-Based Baseline
  Answer Accuracy      : 29.46%
  Equation Accuracy    : 2.27%
  Equation Equivalence : 29.46%
  BLEU Score           : 3.58
  N Samples            : 353


## 3. Template-Based Baseline

Strategy: Learn common equation templates from training data, then match test problems to nearest template based on word overlap.

In [8]:
from collections import defaultdict

def equation_to_template(eq: str) -> str:
    """Replace numbers with NUM placeholder to get structural template."""
    return re.sub(r'\d+\.?\d*', 'NUM', eq)


# Build template library from training set
template_counts = Counter(equation_to_template(x['Equation']) for x in train)
print(f'Unique templates in training set: {len(template_counts)}')
print('\nTop 10 templates:')
for tmpl, cnt in template_counts.most_common(10):
    print(f'  {cnt:4d}  {tmpl}')

Unique templates in training set: 38

Top 10 templates:
   461  X = ( NUM + NUM )
   461  X = ( NUM - NUM )
   176  X = ( NUM * NUM )
   150  X = ( NUM / NUM )
   119  X = ( ( NUM + NUM ) + NUM )
    90  X = ( ( NUM + NUM ) - NUM )
    33  X = ( NUM - ( NUM + NUM ) )
    20  X = ( ( NUM - NUM ) - NUM )
    16  X = ( NUM * ( NUM - NUM ) )
    14  X = ( ( NUM - NUM ) / NUM )


In [10]:
import math

def get_tfidf_keywords(problem_text: str, vocab_idf: dict) -> set:
    """Return set of high-IDF words from problem text."""
    words = set(problem_text.split())
    return {w for w in words if vocab_idf.get(w, 0) > 2.0}


# Build word-to-template index from training set
# For each template, store: template string, example numbers, keyword set
train_records = []
doc_freq = Counter()
for item in train:
    words = set(item['Problem'].split())
    doc_freq.update(words)

N = len(train)
idf = {w: math.log(N / (1 + cnt)) for w, cnt in doc_freq.items()}

for item in train:
    nums = extract_numbers(item['Problem'])
    tmpl = equation_to_template(item['Equation'])
    kws  = get_tfidf_keywords(item['Problem'], idf)
    train_records.append({'numbers': nums, 'template': tmpl, 'keywords': kws,
                          'equation': item['Equation']})


def template_solve(problem: dict, k: int = 3) -> str:
    """Find most keyword-similar training examples and apply their template."""
    text = problem['Problem']
    test_words = set(text.split())
    test_nums  = extract_numbers(text)

    best_score, best_template = -1, None
    for rec in train_records:
        score = len(test_words & rec['keywords'])
        if score > best_score:
            best_score = score
            best_template = rec['template']

    if best_template is None:
        # Fallback: majority template
        best_template = template_counts.most_common(1)[0][0]

    # Fill NUM slots with extracted numbers
    slots = best_template.count('NUM')
    padded = (test_nums + [0] * slots)[:slots]
    result = best_template
    for num in padded:
        result = result.replace('NUM', str(num), 1)
    return result


ex = test[5]
print('Problem:', ex['Problem'])
print('Gold    :', ex['Equation'])
print('Pred    :', template_solve(ex))

Problem: 1 रेस्टॉरंटने जेवणासाठी 9 हॅम्बर्गर बनवले. ते विकले गेले आणि आणखी 3 बनवावे लागले. त्यांनी लंचसाठी किती हॅम्बर्गर बनवले?
Gold    : X = ( 9 + 3 )
Pred    : X = ( 1.0 + 9.0 )


In [11]:
import math
template_preds = [template_solve(x) for x in test]
template_metrics = compute_metrics(gold_test_eqs, template_preds)
print_metrics('Template-Based Baseline', template_metrics)


  Template-Based Baseline
  Answer Accuracy      : 26.91%
  Equation Accuracy    : 1.42%
  Equation Equivalence : 26.91%
  BLEU Score           : 6.06
  N Samples            : 353


## 4. Seq2Seq LSTM Baseline

Character-level encoder → decoder to generate equation string.

**Architecture:**
- Encoder: Bidirectional LSTM over character tokens
- Decoder: LSTM with attention, generates equation character by character
- Trained on 70% Marathi split (~1635 problems)

**Note**: Requires PyTorch. For GPU acceleration, run on Google Colab.

In [12]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

Using device: cpu


In [13]:
# ── Vocabulary ──────────────────────────────────────────────────────────────
class CharVocab:
    PAD, SOS, EOS, UNK = '<PAD>', '<SOS>', '<EOS>', '<UNK>'

    def __init__(self):
        self.ch2idx = {self.PAD: 0, self.SOS: 1, self.EOS: 2, self.UNK: 3}
        self.idx2ch = {v: k for k, v in self.ch2idx.items()}

    def build(self, texts: list):
        chars = sorted(set(ch for t in texts for ch in t))
        for ch in chars:
            if ch not in self.ch2idx:
                idx = len(self.ch2idx)
                self.ch2idx[ch] = idx
                self.idx2ch[idx] = ch
        return self

    def encode(self, text: str, max_len=None) -> list:
        ids = [self.ch2idx.get(c, self.ch2idx[self.UNK]) for c in text]
        if max_len:
            ids = ids[:max_len]
        return ids

    def decode(self, ids: list) -> str:
        return ''.join(self.idx2ch.get(i, self.UNK) for i in ids
                       if i not in (self.ch2idx[self.PAD], self.ch2idx[self.SOS],
                                    self.ch2idx[self.EOS]))

    def __len__(self):
        return len(self.ch2idx)


src_vocab = CharVocab().build([x['Problem']  for x in train + val])
tgt_vocab = CharVocab().build([x['Equation'] for x in train + val])
print(f'Source vocab: {len(src_vocab)}  Target vocab: {len(tgt_vocab)}')

Source vocab: 141  Target vocab: 26


In [14]:
# ── Dataset ─────────────────────────────────────────────────────────────────
MAX_SRC = 200
MAX_TGT = 60


class MWPDataset(Dataset):
    def __init__(self, data, src_vocab, tgt_vocab):
        self.data = data
        self.sv   = src_vocab
        self.tv   = tgt_vocab

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        src  = self.sv.encode(item['Problem'],  MAX_SRC)
        tgt  = self.tv.encode(item['Equation'], MAX_TGT)
        return src, tgt


def collate_fn(batch):
    srcs, tgts = zip(*batch)
    max_src = max(len(s) for s in srcs)
    max_tgt = max(len(t) for t in tgts)
    src_pad = torch.zeros(len(srcs), max_src, dtype=torch.long)
    tgt_pad = torch.zeros(len(tgts), max_tgt + 2, dtype=torch.long)
    src_len = []
    for i, (s, t) in enumerate(zip(srcs, tgts)):
        src_pad[i, :len(s)] = torch.tensor(s)
        tgt_pad[i, 0] = tgt_vocab.ch2idx[tgt_vocab.SOS]
        tgt_pad[i, 1:len(t)+1] = torch.tensor(t)
        tgt_pad[i, len(t)+1]   = tgt_vocab.ch2idx[tgt_vocab.EOS]
        src_len.append(len(s))
    return src_pad, tgt_pad, torch.tensor(src_len)


train_ds = MWPDataset(train, src_vocab, tgt_vocab)
val_ds   = MWPDataset(val,   src_vocab, tgt_vocab)
test_ds  = MWPDataset(test,  src_vocab, tgt_vocab)

train_dl = DataLoader(train_ds, batch_size=64, shuffle=True,  collate_fn=collate_fn)
val_dl   = DataLoader(val_ds,   batch_size=64, shuffle=False, collate_fn=collate_fn)

In [15]:
# ── Model ────────────────────────────────────────────────────────────────────
class Encoder(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden=256, n_layers=2, dropout=0.3):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.rnn = nn.LSTM(embed_dim, hidden, n_layers, batch_first=True,
                           bidirectional=True, dropout=dropout)
        self.fc  = nn.Linear(hidden * 2, hidden)

    def forward(self, src, lengths):
        emb = self.emb(src)
        packed = nn.utils.rnn.pack_padded_sequence(
            emb, lengths.cpu(), batch_first=True, enforce_sorted=False)
        out, (h, c) = self.rnn(packed)
        out, _ = nn.utils.rnn.pad_packed_sequence(out, batch_first=True)
        # Merge bidirectional
        h = torch.cat([h[-2], h[-1]], dim=1)
        h = torch.tanh(self.fc(h)).unsqueeze(0)
        return out, h


class Attention(nn.Module):
    def __init__(self, enc_hid, dec_hid):
        super().__init__()
        self.attn = nn.Linear(enc_hid * 2 + dec_hid, dec_hid)
        self.v    = nn.Linear(dec_hid, 1, bias=False)

    def forward(self, dec_h, enc_out):
        T = enc_out.size(1)
        dec_h = dec_h.squeeze(0).unsqueeze(1).repeat(1, T, 1)
        energy = torch.tanh(self.attn(torch.cat([dec_h, enc_out], dim=2)))
        return torch.softmax(self.v(energy).squeeze(2), dim=1)


class Decoder(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, enc_hid=256, dec_hid=256, dropout=0.3):
        super().__init__()
        self.emb  = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.attn = Attention(enc_hid, dec_hid)
        self.rnn  = nn.LSTM(embed_dim + enc_hid * 2, dec_hid, batch_first=True)
        self.fc   = nn.Linear(dec_hid + enc_hid * 2 + embed_dim, vocab_size)
        self.drop = nn.Dropout(dropout)

    def forward(self, tgt_tok, dec_h, dec_c, enc_out):
        emb = self.drop(self.emb(tgt_tok.unsqueeze(1)))
        a   = self.attn(dec_h, enc_out).unsqueeze(1)
        ctx = torch.bmm(a, enc_out)
        rnn_in = torch.cat([emb, ctx], dim=2)
        out, (h, c) = self.rnn(rnn_in, (dec_h, dec_c))
        pred = self.fc(torch.cat([out.squeeze(1), ctx.squeeze(1), emb.squeeze(1)], dim=1))
        return pred, h, c


class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, tgt_vocab, teacher_forcing=0.5):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.tv = tgt_vocab
        self.tf = teacher_forcing

    def forward(self, src, tgt, lengths):
        B, T = tgt.shape
        vocab_size = len(self.tv)
        outputs = torch.zeros(B, T, vocab_size).to(src.device)

        enc_out, h = self.encoder(src, lengths)
        dec_h = h
        dec_c = torch.zeros_like(dec_h)
        tok   = tgt[:, 0]

        for t in range(1, T):
            pred, dec_h, dec_c = self.decoder(tok, dec_h, dec_c, enc_out)
            outputs[:, t] = pred
            tok = tgt[:, t] if random.random() < self.tf else pred.argmax(1)
        return outputs

    @torch.no_grad()
    def generate(self, src, lengths, max_len=60):
        enc_out, h = self.encoder(src, lengths)
        dec_h = h
        dec_c = torch.zeros_like(dec_h)
        tok   = torch.full((src.size(0),), self.tv.ch2idx[self.tv.SOS],
                            dtype=torch.long, device=src.device)
        result = []
        for _ in range(max_len):
            pred, dec_h, dec_c = self.decoder(tok, dec_h, dec_c, enc_out)
            tok = pred.argmax(1)
            result.append(tok.cpu().tolist())
        return list(zip(*result))  # (B, max_len)


ENC = Encoder(len(src_vocab)).to(DEVICE)
DEC = Decoder(len(tgt_vocab)).to(DEVICE)
MODEL = Seq2Seq(ENC, DEC, tgt_vocab).to(DEVICE)
print(f'Parameters: {sum(p.numel() for p in MODEL.parameters()):,}')

Parameters: 3,660,186


In [ ]:
# ── Training ─────────────────────────────────────────────────────────────────
import time, sys

Path('models').mkdir(exist_ok=True)

N_EPOCHS = 30
optimizer = optim.AdamW(MODEL.parameters(), lr=3e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)
criterion = nn.CrossEntropyLoss(ignore_index=0)

train_losses, val_losses = [], []
n_batches = len(train_dl)

print(f'Training LSTM Seq2Seq on {DEVICE}')
print(f'  Epochs      : {N_EPOCHS}')
print(f'  Train size  : {len(train_ds)}  ({n_batches} batches/epoch)')
print(f'  Val size    : {len(val_ds)}')
print(f'  Parameters  : {sum(p.numel() for p in MODEL.parameters()):,}')
print('-' * 60)

train_start = time.time()

for epoch in range(1, N_EPOCHS + 1):
    epoch_start = time.time()
    MODEL.train()
    total_loss = 0

    for batch_idx, (src, tgt, lengths) in enumerate(train_dl, 1):
        src, tgt, lengths = src.to(DEVICE), tgt.to(DEVICE), lengths
        optimizer.zero_grad()
        out  = MODEL(src, tgt, lengths)
        loss = criterion(out[:, 1:].reshape(-1, len(tgt_vocab)),
                         tgt[:, 1:].reshape(-1))
        loss.backward()
        nn.utils.clip_grad_norm_(MODEL.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()

        # Print batch progress every 5 batches
        if batch_idx % 5 == 0 or batch_idx == n_batches:
            print(f'  Epoch {epoch:2d}/{N_EPOCHS} | Batch {batch_idx:2d}/{n_batches}'
                  f' | Batch Loss: {loss.item():.4f}', end='\r')
            sys.stdout.flush()

    # Validation
    MODEL.eval()
    vloss = 0
    with torch.no_grad():
        for src, tgt, lengths in val_dl:
            src, tgt = src.to(DEVICE), tgt.to(DEVICE)
            out = MODEL(src, tgt, lengths)
            vloss += criterion(out[:, 1:].reshape(-1, len(tgt_vocab)),
                               tgt[:, 1:].reshape(-1)).item()

    avg_train = total_loss / len(train_dl)
    avg_val   = vloss / len(val_dl)
    train_losses.append(avg_train)
    val_losses.append(avg_val)
    scheduler.step(avg_val)

    epoch_secs = time.time() - epoch_start
    elapsed    = time.time() - train_start
    remaining  = elapsed / epoch * (N_EPOCHS - epoch)
    lr_now     = optimizer.param_groups[0]['lr']

    print(f'Epoch {epoch:2d}/{N_EPOCHS} | '
          f'Train: {avg_train:.4f} | Val: {avg_val:.4f} | '
          f'LR: {lr_now:.2e} | '
          f'{epoch_secs:.0f}s/epoch | ETA: {remaining/60:.1f}min')
    sys.stdout.flush()

total_mins = (time.time() - train_start) / 60
print('-' * 60)
print(f'Training complete in {total_mins:.1f} min')
torch.save(MODEL.state_dict(), 'models/lstm_seq2seq.pt')
print('Model saved to models/lstm_seq2seq.pt')

In [ ]:
Path('models').mkdir(exist_ok=True)

# Learning curve
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(train_losses, label='Train Loss')
ax.plot(val_losses,   label='Val Loss')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('LSTM Seq2Seq Training Curve', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('figures/fig_5_lstm_training_curve.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Inference on test set ────────────────────────────────────────────────────
MODEL.eval()
lstm_preds = []

for item in test:
    src_ids = src_vocab.encode(item['Problem'], MAX_SRC)
    src_t   = torch.tensor([src_ids]).to(DEVICE)
    lengths = torch.tensor([len(src_ids)])
    gen     = MODEL.generate(src_t, lengths, max_len=MAX_TGT)[0]
    pred_str = tgt_vocab.decode(gen)
    # Ensure 'X = ' prefix
    if not pred_str.startswith('X'):
        pred_str = 'X = ' + pred_str
    lstm_preds.append(pred_str)

lstm_metrics = compute_metrics(gold_test_eqs, lstm_preds)
print_metrics('LSTM Seq2Seq Baseline', lstm_metrics)

## 5. Baseline Results Comparison

In [ ]:
results = {
    'Rule-Based':    rule_metrics,
    'Template-Based': template_metrics,
    'LSTM Seq2Seq':  lstm_metrics,
}

df_results = pd.DataFrame([
    {
        'Model'               : name,
        'Answer Acc (%)'      : f"{m['answer_accuracy']:.1f}",
        'Equation Acc (%)'    : f"{m['equation_accuracy']:.1f}",
        'Eq Equivalence (%)'  : f"{m['equation_equivalence']:.1f}",
        'BLEU'                : f"{m['bleu']:.1f}",
    }
    for name, m in results.items()
])

print('\nBaseline Results (Test Set)')
print(df_results.to_string(index=False))
df_results.to_csv('figures/table_5_1_baseline_results.csv', index=False)

In [ ]:
# Bar chart for thesis Figure 5.1
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(results))
width = 0.2
metric_keys = ['answer_accuracy', 'equation_accuracy', 'bleu']
metric_labels = ['Answer Acc', 'Equation Acc', 'BLEU']
colors_bar = ['#4C72B0', '#55A868', '#DD8452']

for i, (key, label, color) in enumerate(zip(metric_keys, metric_labels, colors_bar)):
    vals = [results[n][key] for n in results]
    bars = ax.bar(x + i * width, vals, width, label=label, color=color)

ax.set_xticks(x + width)
ax.set_xticklabels(list(results.keys()))
ax.set_ylabel('Score')
ax.set_title('Baseline Model Performance Comparison', fontweight='bold', fontsize=13)
ax.legend()
ax.set_ylim(0, 100)
plt.tight_layout()
plt.savefig('figures/fig_5_1_baseline_comparison.png', bbox_inches='tight')
plt.show()

## 6. Error Analysis — Rule-Based

In [ ]:
error_categories = {
    'Correct': 0,
    'Wrong Operation': 0,
    'Wrong Numbers': 0,
    'Multi-step Failure': 0,
    'No Numbers Found': 0,
}

for item, pred in zip(test, rule_preds):
    gold_ans = evaluate_equation(item['Equation'])
    pred_ans = evaluate_equation(pred)

    if gold_ans is not None and pred_ans is not None and abs(gold_ans - pred_ans) < 1e-3:
        error_categories['Correct'] += 1
    elif item['Number of Operators'] >= 2:
        error_categories['Multi-step Failure'] += 1
    elif len(extract_numbers(item['Problem'])) == 0:
        error_categories['No Numbers Found'] += 1
    else:
        gold_op = classify_operation(item['Equation'])
        pred_op = classify_operation(pred)
        if gold_op != pred_op:
            error_categories['Wrong Operation'] += 1
        else:
            error_categories['Wrong Numbers'] += 1

print('Error Analysis — Rule-Based Baseline')
total = len(test)
for cat, cnt in error_categories.items():
    print(f'  {cat:<25} : {cnt:4d} ({cnt/total*100:.1f}%)')

In [ ]:
# Save all predictions for downstream analysis
preds_out = [
    {
        'pIndex'     : item['pIndex'],
        'Problem'    : item['Problem'],
        'Gold'       : item['Equation'],
        'Rule'       : rule_preds[i],
        'Template'   : template_preds[i],
        'LSTM'       : lstm_preds[i],
    }
    for i, item in enumerate(test)
]
with open('splits/test_predictions_baselines.json', 'w', encoding='utf-8') as f:
    json.dump(preds_out, f, ensure_ascii=False, indent=2)
print('Predictions saved to splits/test_predictions_baselines.json')